# RNA Structure Audit — Quickstart

Evaluate whether an RNA/DNA foundation model encodes genuine secondary structure knowledge.

This notebook uses the **built-in adapters** for 10 pre-configured models.
To evaluate your own model, see [bring_your_own_model.ipynb](bring_your_own_model.ipynb).

**Runtime**: ~10 min per model on a T4 GPU (free Colab). Models up to ~650M params fit on a T4.

In [ ]:
# Install (run once)
!pip install -q rna-structure-audit[all-models]

## Available models

| Adapter | Model | Params | Pretraining | Architecture |
|---------|-------|--------|-------------|-------------|
| `RiNALMoAdapter` | RiNALMo | 650M | RNA | BERT + RoPE + SwiGLU |
| `ERNIERNAAdapter` | ERNIE-RNA | 86M | RNA | BERT |
| `RNAFMAdapter` | RNA-FM | 99M | RNA | BERT |
| `SpliceBERTAdapter` | SpliceBERT | 19M | RNA | BERT |
| `UTRLMAdapter` | UTR-LM | 1.2M | RNA (UTRs) | Transformer encoder |
| `DNABERT2Adapter` | DNABERT-2 | 117M | DNA | BERT (BPE) |
| `NTAdapter` | Nucleotide Transformer v2 | 56M | DNA | ESM + GLU (6-mer) |
| `CaduceusAdapter` | Caduceus | 14M | DNA | BiMamba |
| `HyenaDNAAdapter` | HyenaDNA | 5.4M | DNA | Hyena SSM |
| `EvoAdapter` | Evo | 7B | DNA | StripedHyena (needs A100) |

All adapters auto-download weights from HuggingFace on first use.

In [ ]:
import torch
from rna_structure_audit.adapters import RiNALMoAdapter
from rna_structure_audit.evaluate import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Evaluate a single model

Pick any adapter from the table above. RiNALMo is the strongest performer.

In [ ]:
adapter = RiNALMoAdapter()
results = evaluate(adapter, device=device, output_path="rinalmo_results.json")

report = results["report"]
print(f"Model: {report['model']}")
print(f"Grade: {report.get('grade', 'N/A')} — {report.get('grade_description', '')}")

In [ ]:
if "rung1" in report:
    r1 = report["rung1"]
    print(f"Rung 1 (stem vs loop): {r1['families_exceeding_null']}/{r1['families_scored']} families > null")

if "rung2" in report:
    r2 = report["rung2"]
    print(f"Rung 2 (dinucleotide): {r2['surviving_dinuc']}/{r2['rung1_passing']} survive null")

if "rung3" in report:
    r3 = report["rung3"]
    print(f"Rung 3 (partner specificity): PS = {r3['mean_ps']:.4f}, "
          f"precision = {r3['mean_h3_precision']:.1%}, "
          f"{r3['exceeding_null']}/{r3['gate_passing']} families > null")

## Compare multiple models

Run several models and compare their grades. Smaller models are faster to evaluate.

In [ ]:
from rna_structure_audit.adapters import ERNIERNAAdapter, SpliceBERTAdapter, CaduceusAdapter

models_to_compare = [
    ERNIERNAAdapter(),
    SpliceBERTAdapter(),
    CaduceusAdapter(),
]

comparison = []
for adapter in models_to_compare:
    print(f"\nEvaluating {adapter.name}...")
    result = evaluate(adapter, device=device, output_path=f"{adapter.name.lower()}_results.json")
    r = result["report"]
    comparison.append({
        "model": r["model"],
        "grade": r.get("grade", "?"),
        "ps": r.get("rung3", {}).get("mean_ps", None),
    })

print("\n--- Comparison ---")
for c in comparison:
    ps_str = f"{c['ps']:.4f}" if c["ps"] is not None else "N/A"
    print(f"{c['model']:25s}  Grade: {c['grade']}  PS: {ps_str}")

## Run only Rung 3 (partner specificity)

If you just want the most informative metric, skip Rungs 1-2.

In [ ]:
adapter = ERNIERNAAdapter()
results = evaluate(adapter, device=device, rungs=(3,), output_path="ernierna_rung3.json")

r3 = results["report"].get("rung3", {})
print(f"PS = {r3.get('mean_ps', 0):.4f}")

## Grading scale

| Grade | Meaning |
|-------|---------|
| **A** | Partner-specific: the model knows which position pairs with which |
| **B** | Structure-aware beyond composition: survives dinucleotide controls |
| **C** | Composition-sensitive: stem/loop signal absorbed by nucleotide frequency controls |
| **D** | No detectable structure signal |

The key metric at Rung 3 is **perturbation specificity (PS)**: when you mutate position *i*, does the model preferentially perturb position *i*'s pairing partner over adjacent stem positions? PS > 0.05 with precision > 50% earns an A.

For details, see Tower (2026), *A Three-Rung Evaluation of RNA Structure Awareness in Foundation Models*.